# Notebook 04: Diagnostic pattern at the C3 regime

Section 5.2 of the paper introduces a diagnostic pattern based on
the joint reading of R_certain and p_excl. When R_certain is high
but p_excl is zero, the campaign has very likely reached the
boundary of its discoverable bug surface within the available
time budget. This is not a failure of the triangulation
procedure; it is a signal that the budget is the binding
constraint.

This notebook prints the diagnostic table for the three
campaigns, visualises the diagnostic regions in the (p_excl,
R_certain) plane, and verifies the productivity gain that C3
still exhibits under bug-set parity.


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

DATA_DIR = Path("..") / "data"
with (DATA_DIR / "rq2_discovery.csv").open(newline="") as handle:
    discovery = list(csv.DictReader(handle))

# R_certain values come from the proprietary per-input agreement records.
# The rationale is the same as in scripts/compute_certainty_rates.py:
# joint software hypotheses prevent recovery from the marginal CSV.
R_CERTAIN = {"C2": 0.071, "C3": 0.500, "picoc": 0.246}


## Diagnostic table


In [ ]:
def diagnose(r_certain: float, p_excl: float) -> str:
    if r_certain >= 0.4 and p_excl == 0.0:
        return "BUDGET-LIMITED (extend T)"
    if r_certain < 0.15 and p_excl > 0.5:
        return "CALIBRATION-LIMITED (recalibrate)"
    if r_certain >= 0.2 and p_excl > 0.0:
        return "NOMINAL OPERATION"
    return "INTERMEDIATE"

header = (f"{'Campaign':<10}{'R_certain':>11}{'p_excl':>9}"
          f"{'Prod(EM)':>10}{'Prod(B)':>10}  Diagnosis")
print(header)
print("-" * 80)
for row in discovery:
    camp = row["campaign"]
    r = R_CERTAIN[camp]
    p = float(row["p_excl"])
    prod_em = float(row["prod_crash_em"])
    prod_base = float(row["prod_crash_base"])
    print(f"{camp:<10}{r:>11.3f}{p:>9.2f}{prod_em:>10.2f}{prod_base:>10.2f}  {diagnose(r, p)}")


Expected diagnoses: C2 = INTERMEDIATE, C3 = BUDGET-LIMITED, picoc = NOMINAL.

The CALIBRATION-LIMITED rule guards on ``p_excl > 0.5`` rather than
``p_excl > 0`` because the C2 case has only n = 2 unique bugs in
the union: a single exclusive bug yields p_excl = 0.5 with a
Wilson 95% CI of [0.095, 0.905], which does not constitute strong
evidence of an EM-only regime. C2 is therefore correctly
classified as INTERMEDIATE rather than triggering recalibration.


## Visualisation of diagnostic regions


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.axhspan(0.4, 1.05, xmin=0.0, xmax=0.05 / 1.05,
           color="#ffcccc", alpha=0.5, label="BUDGET-LIMITED")
ax.axvspan(0.0, 1.05, ymin=0.0, ymax=0.15 / 1.05, color="none")
ax.add_patch(plt.Rectangle((0.0, 0.0), 0.05, 0.15,
                            color="#ffcccc", alpha=0.5))
ax.add_patch(plt.Rectangle((0.0, 0.0), 1.05, 0.15,
                            color="#fff0cc", alpha=0.4,
                            label="CALIBRATION-LIMITED candidate band"))
ax.add_patch(plt.Rectangle((0.0, 0.4), 0.05, 0.65,
                            color="#ffcccc", alpha=0.5))

for row in discovery:
    camp = row["campaign"]
    r = R_CERTAIN[camp]
    p = float(row["p_excl"])
    ax.scatter(p, r, s=90, color="#1f4e79", zorder=3)
    ax.annotate(camp, (p, r), xytext=(8, 6), textcoords="offset points")

ax.set_xlim(-0.02, 1.0)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel("p_excl")
ax.set_ylabel("R_certain")
ax.set_title("Diagnostic regions in the (p_excl, R_certain) plane")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
fig.tight_layout()
plt.show()


## Productivity verification under parity (C3)


In [ ]:
row_c3 = next(r for r in discovery if r["campaign"] == "C3")
prod_em = float(row_c3["prod_crash_em"])
prod_base = float(row_c3["prod_crash_base"])
gain_pct = (prod_em - prod_base) / prod_base * 100
print(f"C3 productivity: EM = {prod_em:.2f}, baseline = {prod_base:.2f}")
print(f"Relative gain: {gain_pct:.1f}%")


Expected: 30.3%. This matches the value reported in the paper for
C3 under bug-set parity, demonstrating that EM-guided seed
selection retains practical value even when no exclusive bugs
are surfaced, by allocating fuzzing time to higher-yield seeds.


## Practitioner summary

| Aspect | Content |
|---|---|
| Observation | High R_certain combined with p_excl = 0. |
| Likely cause | The bug surface reachable within budget T has been exhausted. The cross-modal pipeline is performing as designed. |
| Recommended action | Extend the time budget rather than altering the seed-selection criterion. EM-guided selection still buys productivity even at parity. |
